# Processamento de Imagens

## O que é?

**Processamento de Imagens** é a área que realiza **operações sobre imagens existentes** para melhorá-las, transformá-las ou extrair características básicas. O fluxo é: **Imagem → Operação matemática → Imagem modificada** (ou métrica).

## Características Principais

- **Entrada:** imagem real (foto, scan, frame de vídeo)
- **Saída:** imagem processada ou dados numéricos
- **Operações típicas:** filtragem espacial, transformadas de frequência, morfologia, segmentação por limiar
- **Aplicações:** edição de fotos, diagnóstico médico, controle de qualidade industrial

## Aplicação escolhida: Filtros Espaciais e Detecção de Bordas

Vamos demonstrar três técnicas clássicas:

1. **Filtragem espacial** (suavização / convolução)
2. **Detecção de bordas** (Sobel e Canny)
3. **Equalização de histograma** (melhoria de contraste)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import data, exposure, filters
from skimage.color import rgb2gray

## 1. Imagem de Referência

Usamos a imagem clássica `camera` do scikit-image (fotógrafo), amplamente utilizada em literatura de processamento de imagens.

In [ ]:
# Imagem em escala de cinza (512x512)
img_gray = data.camera()

# Imagem colorida para demonstração de equalização
img_color = data.astronaut()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title(f'Imagem Original (cinza) — {img_gray.shape}', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_color)
axes[1].set_title(f'Imagem Colorida — {img_color.shape}', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()
print(f"Dtype: {img_gray.dtype} | Valores: [{img_gray.min()}, {img_gray.max()}]")

## 2. Filtragem Espacial — Convolução

A operação de convolução aplica um **kernel** (matriz de pesos) deslizando sobre a imagem:

$$g(x,y) = \sum_{s=-a}^{a} \sum_{t=-b}^{b} f(x+s, y+t) \cdot h(s,t)$$

- **Kernel Gaussian:** pesos decaem com distância → suaviza e remove ruído
- **Kernel de média:** todos os pesos iguais → suavização mais agressiva

In [ ]:
# Adicionamos ruído para demonstrar o efeito da filtragem
np.random.seed(42)
ruido = np.random.normal(0, 25, img_gray.shape).astype(np.float32)
img_ruidosa = np.clip(img_gray.astype(np.float32) + ruido, 0, 255).astype(np.uint8)

# Filtros de suavização
blur_media   = cv2.blur(img_ruidosa, (7, 7))                  # Média
blur_gaussian = cv2.GaussianBlur(img_ruidosa, (7, 7), sigmaX=2)  # Gaussiano
blur_mediana  = cv2.medianBlur(img_ruidosa, 7)                 # Mediana (robusto a sal-e-pimenta)

# Kernels
kernel_media = np.ones((7,7), np.float32) / 49
kernel_gaussian_1d = cv2.getGaussianKernel(7, 2)
kernel_gaussian_2d = kernel_gaussian_1d @ kernel_gaussian_1d.T

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

imgs = [img_gray, img_ruidosa, blur_media, blur_gaussian, blur_mediana]
titles = ['Original', 'Com ruído Gaussian', 'Filtro Média (7×7)', 'Filtro Gaussiano (7×7)', 'Filtro Mediana (7×7)']

for i, (im, t) in enumerate(zip(imgs[:4], titles[:4])):
    axes[0, i].imshow(im, cmap='gray')
    axes[0, i].set_title(t, fontsize=10)
    axes[0, i].axis('off')

axes[1, 0].imshow(blur_mediana, cmap='gray')
axes[1, 0].set_title('Filtro Mediana (7×7)', fontsize=10)
axes[1, 0].axis('off')

# Visualização dos kernels
im1 = axes[1, 1].imshow(kernel_media, cmap='Blues', interpolation='nearest')
axes[1, 1].set_title('Kernel Média', fontsize=10)
plt.colorbar(im1, ax=axes[1,1], fraction=0.046)

im2 = axes[1, 2].imshow(kernel_gaussian_2d, cmap='Oranges', interpolation='nearest')
axes[1, 2].set_title('Kernel Gaussiano 2D', fontsize=10)
plt.colorbar(im2, ax=axes[1,2], fraction=0.046)

# PSNR para comparação de qualidade
def psnr(original, processada):
    mse = np.mean((original.astype(float) - processada.astype(float))**2)
    return 20 * np.log10(255.0 / np.sqrt(mse)) if mse > 0 else float('inf')

psnr_vals = {
    'Ruidosa':  psnr(img_gray, img_ruidosa),
    'Média':    psnr(img_gray, blur_media),
    'Gaussiano': psnr(img_gray, blur_gaussian),
    'Mediana':  psnr(img_gray, blur_mediana),
}
cores = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
axes[1, 3].bar(psnr_vals.keys(), psnr_vals.values(), color=cores)
axes[1, 3].set_title('PSNR (dB) — quanto maior, melhor', fontsize=10)
axes[1, 3].set_ylabel('dB')
for bar, val in zip(axes[1,3].patches, psnr_vals.values()):
    axes[1,3].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   f'{val:.1f}', ha='center', fontsize=9)

plt.suptitle('Filtragem Espacial — Remoção de Ruído', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Detecção de Bordas

Bordas correspondem a regiões de **variação brusca de intensidade**. São detectadas pelo gradiente da imagem.

### Operador de Sobel

Calcula o gradiente por convolução com dois kernels direccionais:

$$G_x = \begin{bmatrix}-1&0&+1\\-2&0&+2\\-1&0&+1\end{bmatrix} * I \qquad G_y = \begin{bmatrix}-1&-2&-1\\0&0&0\\+1&+2&+1\end{bmatrix} * I$$

$$|G| = \sqrt{G_x^2 + G_y^2}$$

### Algoritmo de Canny

Pipeline multi-etapa: suavização → gradiente Sobel → supressão de não-máximos → limiarização por histerese.

In [ ]:
img_suave = cv2.GaussianBlur(img_gray, (5, 5), sigmaX=1)

# Sobel
sobel_x = cv2.Sobel(img_suave, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(img_suave, cv2.CV_64F, 0, 1, ksize=3)
sobel_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
sobel_angulo    = np.arctan2(sobel_y, sobel_x) * 180 / np.pi

# Canny
canny_strict = cv2.Canny(img_gray, threshold1=50,  threshold2=150)
canny_loose  = cv2.Canny(img_gray, threshold1=20,  threshold2=80)

# Laplaciano (segunda derivada)
laplacian = cv2.Laplacian(img_suave, cv2.CV_64F)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes[0,0].imshow(img_gray, cmap='gray')
axes[0,0].set_title('Original', fontsize=11)

axes[0,1].imshow(np.abs(sobel_x), cmap='gray')
axes[0,1].set_title('Sobel Gx (borda vertical)', fontsize=11)

axes[0,2].imshow(np.abs(sobel_y), cmap='gray')
axes[0,2].set_title('Sobel Gy (borda horizontal)', fontsize=11)

axes[0,3].imshow(sobel_magnitude, cmap='gray')
axes[0,3].set_title('Magnitude |∇I| = √(Gx²+Gy²)', fontsize=11)

im_ang = axes[1,0].imshow(sobel_angulo, cmap='hsv')
axes[1,0].set_title('Ângulo do Gradiente', fontsize=11)
plt.colorbar(im_ang, ax=axes[1,0], fraction=0.046, label='graus')

axes[1,1].imshow(canny_strict, cmap='gray')
axes[1,1].set_title('Canny (50, 150) — estrito', fontsize=11)

axes[1,2].imshow(canny_loose, cmap='gray')
axes[1,2].set_title('Canny (20, 80) — permissivo', fontsize=11)

axes[1,3].imshow(np.abs(laplacian), cmap='gray')
axes[1,3].set_title('Laplaciano (2ª derivada)', fontsize=11)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Detecção de Bordas — Gradiente da Imagem', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Equalização de Histograma

Redistribui os valores de intensidade para **ocupar toda a faixa disponível**, aumentando o contraste. Utiliza a função de distribuição acumulada (CDA) do histograma como mapeamento.

In [ ]:
# Imagem com baixo contraste (simulada)
img_baixo_contraste = (img_gray * 0.4 + 60).astype(np.uint8)

# Equalização global
img_equalizada = cv2.equalizeHist(img_baixo_contraste)

# CLAHE (Contrast Limited Adaptive Histogram Equalization) — local
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img_baixo_contraste)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

imagens = [img_baixo_contraste, img_equalizada, img_clahe]
titulos = ['Baixo Contraste (original)', 'Equalização Global', 'CLAHE (Equalização Adaptativa)']

for col, (im, titulo) in enumerate(zip(imagens, titulos)):
    axes[0, col].imshow(im, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(titulo, fontsize=11)
    axes[0, col].axis('off')

    hist, bins = np.histogram(im.flatten(), bins=256, range=[0, 256])
    cda = np.cumsum(hist) / hist.sum()

    color_hist = '#e74c3c' if col == 0 else ('#2ecc71' if col == 1 else '#3498db')
    axes[1, col].bar(range(256), hist, color=color_hist, alpha=0.6, label='Histograma')
    ax2 = axes[1, col].twinx()
    ax2.plot(cda, color='black', lw=1.5, label='CDA')
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('CDA', fontsize=9)
    axes[1, col].set_xlabel('Intensidade (0-255)')
    axes[1, col].set_ylabel('Frequência')
    axes[1, col].set_title(f'Histograma — {titulo}', fontsize=10)

plt.suptitle('Equalização de Histograma — Melhoria de Contraste', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Pipeline Completo de Processamento

In [ ]:
# Pipeline: Imagem ruidosa → Suavização → Equalização → Detecção de Bordas
etapa1 = img_ruidosa
etapa2 = cv2.GaussianBlur(etapa1, (5,5), 1)
etapa3 = cv2.equalizeHist(etapa2)
etapa4 = cv2.Canny(etapa3, 30, 100)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
etapas = [etapa1, etapa2, etapa3, etapa4]
nomes  = ['1. Entrada (ruidosa)', '2. Suavização\nGaussiana', '3. Equalização\nde Histograma', '4. Canny\n(detecção de bordas)']

for ax, img, nome in zip(axes, etapas, nomes):
    ax.imshow(img, cmap='gray')
    ax.set_title(nome, fontsize=11)
    ax.axis('off')

# Setas indicando o fluxo
for i in range(3):
    fig.text(0.255 + i*0.25, 0.5, '→', fontsize=20, ha='center', va='center')

plt.suptitle('Pipeline de Processamento de Imagens', fontsize=13)
plt.tight_layout()
plt.show()

## Conclusão

**Processamento de Imagens** opera sobre imagens já existentes, transformando-as matematicamente:

| Aspecto | Processamento de Imagens |
|---------|-------------------------|
| **Entrada** | Imagem real (foto, scan) |
| **Saída** | Imagem modificada ou métricas |
| **Operação** | Transformações matemáticas pixel a pixel ou por convolução |
| **Exemplos** | Filtros do Instagram, diagnóstico médico por imagem |

Diferentemente da Síntese de Imagens (que **cria** imagens do zero), o Processamento parte de imagens reais. Diferentemente da Visão Computacional, o objetivo não é **entender** a imagem, mas **melhorá-la** ou **transformá-la**.